# Clase 09 · Mejora y evaluación del RAG
**Inteligencia Artificial Generativa y Sistemas Inteligentes · EAM 2026-II**  
**Profesor: Robert Garcia Rey**

## Propósito
Vamos a comprobar cómo cambia la evidencia cuando añadimos **BM25, fusión RRF y reranking** al RAG. Conservamos el corpus, los embeddings, el prompt y el modelo generativo de la clase anterior. Comparamos **A: búsqueda semántica**, **B: búsqueda híbrida con BM25 + semántica + RRF** y **C: la misma búsqueda híbrida con reranking**. Cada alternativa entrega hasta tres pasajes. Una modificación puede mejorar, empatar o empeorar. No adelantamos el resultado.

Trabajaremos en **Colab con CPU**. Conviene ejecutar la instalación y cargar los modelos antes del encuentro. El reranker se descarga una vez en cada entorno nuevo y no requiere otra API. **Supabase realiza la búsqueda semántica; BM25, RRF y el reranker se ejecutan en Colab; Groq genera.** La búsqueda léxica usa todos los fragmentos autorizados del corpus, no solo los candidatos semánticos.

Necesitamos la tabla y la función de la clase anterior, con el corpus ya cargado. También necesitamos **`Clase_08_Evidencia_RAG.json`**, descargado en el punto 10 de aquel notebook, y estos secretos de Colab con permiso de lectura: `SUPABASE_URL`, `SUPABASE_PUBLISHABLE_KEY`, `SUPABASE_EMAIL`, `SUPABASE_PASSWORD` y `GROQ_API_KEY`. Si falta el JSON, lo recuperamos ejecutando ese punto de la práctica anterior. **No subimos otro ZIP, no volvemos a fragmentar y no ejecutamos SQL nuevo.** Este notebook solo lee Supabase.

Usaremos los manuales **ficticios** de acceso y constancias. En el punto 4 fijamos las preguntas y sus evidencias antes de ver el ranking. En el punto 8 ingresamos el texto del usuario. La evaluación de recuperación no consume Groq. La comparación de respuestas realiza **dos llamadas**, que podemos desactivar. Solo trabajaremos con textos autorizados, sin expedientes ni datos personales reales.

## 1. Preparación
Mantenemos las versiones de la práctica anterior. Si Colab pide reiniciar después de instalar, lo hacemos y continuamos desde este punto.

In [1]:
%pip -q install sentence-transformers==6.1.0 supabase==2.32.0 groq==1.7.0 pydantic==2.13.5 rank-bm25==0.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [2]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
from typing import Literal
from urllib.parse import urlparse
import hashlib, json, re, unicodedata, zipfile
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from supabase import create_client, ClientOptions
from groq import Groq
from pydantic import BaseModel, ConfigDict
from google.colab import userdata, files

K = 3
M = 10
C_RRF = 60
BM25_K1, BM25_B, BM25_EPSILON = 1.5, 0.75, 0.25
OPCIONES = ("A_semantica", "B_hibrida_RRF", "C_hibrida_reranking")
MODELO_RERANKER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
REVISION_RERANKER = "1427fd652930e4ba29e8149678df786c240d8825"
FECHA = datetime.now(timezone.utc).isoformat()

## 2. Recuperamos la configuración y abrimos la sesión
Subimos **solo el JSON de evidencia de la clase anterior**. Usamos su identificador de corpus, modelo, revisión y prompt. No reutilizamos las respuestas anteriores como respuestas esperadas. La contraseña corresponde al usuario de **Supabase Auth**, no al panel ni a PostgreSQL. Conservamos la clave publicable y los permisos de cada propietario.

In [4]:
subidos = files.upload()
if len(subidos) != 1:
    raise ValueError("Sube únicamente Clase_08_Evidencia_RAG.json.")
contenido = next(iter(subidos.values()))
if len(contenido) > 5_000_000:
    raise ValueError("El JSON supera los 5 MB previstos para esta práctica.")
anterior = json.loads(contenido)
CORPUS_ID = anterior.get("corpus_id", "")
configuracion = anterior.get("configuracion_embeddings", {})
PROMPT_SISTEMA = anterior.get("prompt_sistema", "")
PROMPT_VERSION = anterior.get("prompt_version", "")
MODELO_GENERATIVO = anterior.get("modelo_generativo", "")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", CORPUS_ID):
    raise ValueError("El JSON no contiene un corpus válido de la práctica anterior.")
if configuracion.get("modelo") != "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2":
    raise ValueError("Esta continuación conserva el modelo de embeddings de la clase anterior.")
if not re.fullmatch(r"[0-9a-f]{40}", configuracion.get("revision", "")):
    raise ValueError("Falta la revisión exacta de los embeddings.")
if configuracion.get("normalizados") is not True or configuracion.get("similitud") != "coseno":
    raise ValueError("Revisemos la representación del índice anterior.")
if not PROMPT_VERSION or not isinstance(PROMPT_SISTEMA, str) or not 100 <= len(PROMPT_SISTEMA) <= 10000:
    raise ValueError("Falta el prompt del RAG anterior.")
if MODELO_GENERATIVO != "openai/gpt-oss-20b":
    raise ValueError("Esta comparación mantiene el modelo generativo del demo anterior.")

def secreto(nombre):
    try:
        valor = userdata.get(nombre)
    except Exception:
        raise RuntimeError(f"Revisa el secreto {nombre} y su permiso en Colab.") from None
    if not valor:
        raise RuntimeError(f"El secreto {nombre} está vacío.")
    return valor

url = secreto("SUPABASE_URL").rstrip("/")
clave = secreto("SUPABASE_PUBLISHABLE_KEY")
if urlparse(url).scheme != "https" or not (urlparse(url).hostname or "").endswith(".supabase.co"):
    raise ValueError("Usa la Project URL HTTPS de Supabase.")
if not clave.startswith("sb_publishable_"):
    raise ValueError("Usa una clave publicable, nunca una clave administrativa.")
supabase = create_client(url, clave, options=ClientOptions(persist_session=False))
try:
    sesion = supabase.auth.sign_in_with_password({
        "email": secreto("SUPABASE_EMAIL"), "password": secreto("SUPABASE_PASSWORD")})
except Exception:
    raise RuntimeError("Revisa el usuario de Auth, su confirmación y la conexión.") from None
if sesion.user is None or sesion.session is None:
    raise RuntimeError("Supabase no devolvió una sesión válida.")
USUARIO_ID = sesion.user.id
print("Sesión iniciada. Corpus seleccionado:", CORPUS_ID)

Saving Clase_08_Evidencia_RAG.json to Clase_08_Evidencia_RAG (1).json
Sesión iniciada. Corpus seleccionado: analizador_documental-d9076c624ed4


## 3. Revisamos el corpus persistido y cargamos los modelos
Leemos los fragmentos del mismo propietario, corpus, modelo y revisión. El límite de 500 mantiene acotada esta práctica. Conservamos los embeddings existentes. El **cross-encoder multilingüe** recibe cada pareja de pregunta y pasaje para reordenarla. No vuelve a crear el índice ni añade documentos.

La carga inicial y el calentamiento quedan fuera de las mediciones. El límite del reranker se revisa aparte del límite de los embeddings. Si una entrada no cabe, detenemos esa prueba en lugar de truncarla sin avisar. **BM25 en memoria es una decisión del laboratorio:** si el corpus supera 500 fragmentos, necesitamos otro diseño de índice, no recortar silenciosamente los documentos.

In [5]:
# Cargar los fragmentos de la base de datos Supabase
lectura = (supabase.table("clase08_fragmentos")
    .select("chunk_id,content,metadata")
    .eq("owner_id", USUARIO_ID).eq("corpus_id", CORPUS_ID)
    .eq("configuracion->>modelo", configuracion["modelo"])
    .eq("configuracion->>revision", configuracion["revision"])
    .order("chunk_id").limit(501).execute())

fragmentos = lectura.data
if not 1 <= len(fragmentos) <= 500:
    raise ValueError("Se esperaban entre 1 y 500 fragmentos en el corpus.")

por_id = {r["chunk_id"]: r for r in fragmentos}

display(pd.DataFrame([{"id": r["chunk_id"], "fuente": r["metadata"]["source"],
                       "sección": r["metadata"]["section"], "texto": r["content"]}
                      for r in fragmentos]).head(10))

# Cargar modelos en CPU para evaluadores y reranking
modelo = SentenceTransformer(configuracion["modelo"], revision=configuracion["revision"], device="cpu")
reranker = CrossEncoder(MODELO_RERANKER, revision=REVISION_RERANKER, device="cpu", max_length=512)

# Calentamiento
modelo.encode("consulta de prueba", normalize_embeddings=True)
reranker.predict([("factura", "Requisitos de contabilizacion de factura")], show_progress_bar=False)

print(f"Corpus listo: {len(fragmentos)} fragmentos. Recuperación fijada en M={M} candidatos y K={K} finales.")

,id,fuente,sección,texto
0,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,Contratacion legal,Guia de Contratacion e Identificacion / Contra...
1,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,Contratacion legal,Guia de Contratacion e Identificacion / Contra...
2,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,Contratacion legal,Guia de Contratacion e Identificacion / Contra...
3,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,Identificaciones e imagen borrosa,Guia de Contratacion e Identificacion / Identi...
4,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,Identificaciones e imagen borrosa,Guia de Contratacion e Identificacion / Identi...
5,politicas_facturacion:1.0-documental:parrafos:5,politicas_facturacion.md,Requisitos de contabilizacion,Manual de Politicas de Facturacion / Requisito...
6,politicas_facturacion:1.0-documental:parrafos:6,politicas_facturacion.md,Requisitos de contabilizacion,Manual de Politicas de Facturacion / Requisito...
7,politicas_facturacion:1.0-documental:parrafos:7,politicas_facturacion.md,Requisitos de contabilizacion,Manual de Politicas de Facturacion / Requisito...
8,politicas_facturacion:1.0-documental:parrafos:8,politicas_facturacion.md,Codigos de seguimiento,Manual de Politicas de Facturacion / Codigos d...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (15 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Corpus listo: 9 fragmentos. Recuperación fijada en M=10 candidatos y K=3 finales.


## 4. Fijamos las preguntas y la evidencia esperada
Antes de buscar leemos los originales y definimos qué información necesita cada respuesta. Las frases siguientes **localizan fragmentos del corpus didáctico**, no evalúan por coincidencia de palabras la respuesta del modelo. La negación y la condición completa deben aparecer en el fragmento identificado. Si cambiamos el corpus, reemplazamos estas anotaciones y verificamos sus identificadores.

Cada unidad de información admite uno o varios fragmentos que la respaldan. La unión de esos identificadores es el conjunto relevante para Precision y Recall. La cobertura comprueba cuántas unidades necesarias llegaron al contexto. Dos unidades pueden estar en el mismo fragmento.

Usamos cuatro preguntas de **desarrollo**, dos de **reserva** y dos de **comportamiento**. Decidimos la configuración con desarrollo. Abrimos la reserva una sola vez al cerrar y no la usamos para ajustar otra vez. Los casos sin evidencia documental esperada quedan fuera de las métricas de recuperación y se revisan por su respuesta. Este conjunto pequeño enseña el procedimiento, no certifica el proyecto. Cada equipo lo ampliará con sus casos.

In [8]:
# 4. Fijamos las preguntas y la evidencia esperada
def normalizar(texto):
    return " ".join(unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower().split())

def evidencia(frase, fuente):
    ids = [r["chunk_id"] for r in fragmentos
           if r["metadata"]["source"] == fuente and normalizar(frase) in normalizar(r["content"])]
    if not ids:
        raise ValueError(f"Revisa el original y anota la evidencia del proyecto: {fuente}, '{frase}'")
    return ids

# Casos ajustados y validados contra el corpus del Analizador de Documentos
casos = [
    # --- DESARROLLO ---
    {
        "id": "D1",
        "grupo": "desarrollo",
        "pregunta": "¿Qué requisitos se necesitan para contabilizar una factura?",
        "unidades": {"requisitos contabilizacion": evidencia("NIT o NIF del emisor", "politicas_facturacion.md")},
        "esperado": "Incluir NIT/NIF del emisor, fecha de expedición, desglose de impuestos y total a pagar superior a $0.00 USD.",
        "estados": ["respondida"]
    },
    {
        "id": "D2",
        "grupo": "desarrollo",
        "pregunta": "¿Se contabilizan recibos o facturas con tachaduras o cuya fecha sea mayor a 90 días?",
        "unidades": {"rechazo por tachaduras o vigencia": evidencia("con tachaduras o cuya fecha de emision sea mayor a 90 dias", "politicas_facturacion.md")},
        "esperado": "No se contabilizarán recibos o facturas con tachaduras o cuya fecha de emisión sea mayor a 90 días calendario.",
        "estados": ["respondida"]
    },
    {
        "id": "D3",
        "grupo": "desarrollo",
        "pregunta": "¿Qué pasa si una cédula o documento de identidad tiene la fotografía borrosa?",
        "unidades": {"imagen ilegible o borrosa": evidencia("fotografia o la fecha de expedicion borrosa", "gestion_contratos_identificacion.md")},
        "esperado": "La solicitud permanece pendiente de información y el expediente se marca como rechazado por imagen ilegible.",
        "estados": ["respondida"]
    },
    {
        "id": "D4",
        "grupo": "desarrollo",
        "pregunta": "Si un usuario solicita alterar o cancelar un contrato fuera del procedimiento, ¿qué hace el sistema?",
        "unidades": {"alteracion fuera del procedimiento": evidencia("fuera del procedimiento", "gestion_contratos_identificacion.md")},
        "esperado": "El sistema debe marcar la petición como requiere_revision_humana=True.",
        "estados": ["respondida"]
    },

    # --- RESERVA ---
    {
        "id": "R1",
        "grupo": "reserva",
        "pregunta": "¿Dónde se consulta el estado de procesamiento de una factura registrada?",
        "unidades": {"modulo central de seguimiento": evidencia("modulo central mediante su codigo", "politicas_facturacion.md")},
        "esperado": "Se consulta en el módulo central mediante su código asignado con formato DOC-XXXX.",
        "estados": ["respondida"]
    },
    {
        "id": "R2",
        "grupo": "reserva",
        "pregunta": "¿Cuál es el estado actual de la factura con código DOC-1001?",
        "unidades": {"no contiene estados individuales": evidencia("no contiene los estados de las facturas individuales", "politicas_facturacion.md")},
        "esperado": "Explicar la consulta en el módulo central y aclarar que el manual normativo no contiene los estados individuales.",
        "estados": ["parcial", "evidencia_insuficiente"]
    },

    # --- COMPORTAMIENTO ---
    {
        "id": "C1",
        "grupo": "comportamiento",
        "pregunta": "¿Cómo lo hago?",
        "unidades": {},
        "esperado": "Pedir aclaración sobre el trámite o documento específico.",
        "estados": ["aclaracion"]
    },
    {
        "id": "C2",
        "grupo": "comportamiento",
        "pregunta": "¿Cuáles son los requisitos para solicitar una beca?",
        "unidades": {},
        "esperado": "Reconocer que la evidencia no contiene información sobre becas.",
        "estados": ["evidencia_insuficiente"]
    },
]

display(pd.DataFrame([{"caso": c["id"], "grupo": c["grupo"], "pregunta": c["pregunta"],
                       "respuesta esperada": c["esperado"]} for c in casos]))

,caso,grupo,pregunta,respuesta esperada
0,D1,desarrollo,¿Qué requisitos se necesitan para contabilizar...,"Incluir NIT/NIF del emisor, fecha de expedició..."
1,D2,desarrollo,¿Se contabilizan recibos o facturas con tachad...,No se contabilizarán recibos o facturas con ta...
2,D3,desarrollo,¿Qué pasa si una cédula o documento de identid...,La solicitud permanece pendiente de informació...
3,D4,desarrollo,Si un usuario solicita alterar o cancelar un c...,El sistema debe marcar la petición como requie...
4,R1,reserva,¿Dónde se consulta el estado de procesamiento ...,Se consulta en el módulo central mediante su c...
5,R2,reserva,¿Cuál es el estado actual de la factura con có...,Explicar la consulta en el módulo central y ac...
6,C1,comportamiento,¿Cómo lo hago?,Pedir aclaración sobre el trámite o documento ...
7,C2,comportamiento,¿Cuáles son los requisitos para solicitar una ...,Reconocer que la evidencia no contiene informa...


## 5. Construimos la recuperación híbrida
Usamos la misma pregunta y el mismo corpus para los tres recorridos. **A → B** permite observar el efecto de añadir BM25 y RRF. **B → C** permite observar el efecto del reranker sobre exactamente los mismos candidatos híbridos. No cambiamos el prompt ni el generador para favorecer una opción.

### 5.1. Índice léxico BM25
BM25 valora las palabras de la pregunta en cada fragmento: combina su frecuencia, lo discriminativas que son en el corpus y la longitud del fragmento. Construimos el índice **una vez sobre todos los fragmentos del punto 3**. Usamos `BM25Okapi` de `rank-bm25`, con sus parámetros registrados. Esta implementación aplica un ajuste a IDF; no interpretamos sus puntajes como probabilidades.

Aplicamos la misma tokenización al corpus y a la consulta: normalización Unicode y minúsculas; conservamos tildes, negaciones y códigos como `SOL-1001`. No hacemos stemming ni eliminamos palabras comunes. Es una línea base transparente, no una configuración óptima para todos los proyectos. Un documento sin ninguna palabra de la consulta **no entra en la lista BM25**, aunque haya posiciones disponibles. Puede haber coincidencias léxicas con puntaje cero o negativo según IDF; la ausencia de coincidencias se comprueba con los tokens, no con un umbral de puntaje.

In [9]:
def tokenizar_bm25(texto):
    return re.findall(r"[^\W_]+(?:[-_][^\W_]+)*", unicodedata.normalize("NFKC", texto).casefold())

corpus_tokens = [tokenizar_bm25(r["content"]) for r in fragmentos]
indice_bm25 = BM25Okapi(corpus_tokens, k1=BM25_K1, b=BM25_B, epsilon=BM25_EPSILON)

def buscar_bm25(consulta):
    inicio = perf_counter()
    tokens = tokenizar_bm25(consulta)
    puntajes = np.asarray(indice_bm25.get_scores(tokens), dtype=float)
    coincidencias = [i for i, palabras in enumerate(corpus_tokens) if set(tokens).intersection(palabras)]
    orden = sorted(coincidencias, key=lambda i: (-puntajes[i], fragmentos[i]["chunk_id"]))[:M]
    salida = [dict(fragmentos[i], puntaje_bm25=float(puntajes[i])) for i in orden]
    return salida, (perf_counter() - inicio) * 1000

print("Índice BM25:", len(corpus_tokens), "fragmentos. Ejemplo de tokens:", tokenizar_bm25("Error 403 en SOL-1001; no está emitida."))

Índice BM25: 9 fragmentos. Ejemplo de tokens: ['error', '403', 'en', 'sol-1001', 'no', 'está', 'emitida']


### 5.2. Fusión de rankings con RRF
Recuperamos hasta **M = 10** resultados por buscador y fusionamos su **unión por `chunk_id`**. Un fragmento que solo encontró BM25 también puede entrar. RRF suma `1 / (60 + posición)` por lista; las posiciones empiezan en uno. Si el fragmento no aparece en una lista, su aporte es cero. **60 suaviza el peso de las primeras posiciones; no es la cantidad de resultados.** No sumamos similitud coseno y puntaje BM25.

Ordenamos la unión por el total RRF, resolviendo empates por identificador para hacer explícita la regla. Conservamos hasta diez candidatos híbridos para que el reranker tenga un trabajo acotado. B toma los tres primeros de esa lista; C reordena **esa misma lista de hasta diez** y toma tres. Un reranker no puede recuperar evidencia que quedó fuera. En el punto 8 veremos las posiciones, los dos aportes y su suma.

In [10]:
def fusionar_rrf(semanticos, lexicos, c=C_RRF):
    inicio = perf_counter()
    union = {}
    for nombre, lista in [("semantica", semanticos), ("bm25", lexicos)]:
        for posicion, r in enumerate(lista, 1):
            registro = union.setdefault(r["chunk_id"], dict(r, posicion_semantica=None,
                posicion_bm25=None, aporte_semantica=0.0, aporte_bm25=0.0))
            registro.update(r)
            registro[f"posicion_{nombre}"] = posicion
            registro[f"aporte_{nombre}"] = 1 / (c + posicion)
    for registro in union.values():
        registro["puntaje_rrf"] = registro["aporte_semantica"] + registro["aporte_bm25"]
    fusionados = sorted(union.values(), key=lambda r: (-r["puntaje_rrf"], r["chunk_id"]))
    return fusionados, (perf_counter() - inicio) * 1000

### 5.3. Búsqueda semántica y reranking
La recuperación semántica conserva la función de Supabase de la clase anterior, sus filtros y sus embeddings. Validamos los candidatos contra la lectura del punto 3. El cross-encoder recibe pregunta y fragmento; su puntaje sirve para ordenar, **no se suma a RRF ni es una probabilidad de acierto**.

In [11]:
def recuperar(consulta):
    inicio = perf_counter()
    vector = modelo.encode(consulta, normalize_embeddings=True)
    candidatos = supabase.rpc("clase08_buscar_fragmentos", {
        "query_embedding": vector.tolist(), "p_corpus_id": CORPUS_ID,
        "p_modelo": configuracion["modelo"], "p_revision": configuracion["revision"],
        "match_count": M}).execute().data
    tiempo = (perf_counter() - inicio) * 1000
    return candidatos, tiempo

def reordenar(consulta, candidatos):
    pares = [(consulta, r["content"]) for r in candidatos]
    inicio = perf_counter()
    puntajes = np.asarray(reranker.predict(pares, show_progress_bar=False), dtype=float).reshape(-1)
    orden = sorted(range(len(candidatos)), key=lambda i: (-puntajes[i], candidatos[i]["chunk_id"]))
    salida = [dict(candidatos[i], puntaje_reranker=float(puntajes[i])) for i in orden]
    return salida, (perf_counter() - inicio) * 1000

## 6. Medimos la recuperación en desarrollo
**Precision@3** divide los fragmentos relevantes recuperados entre tres. **Recall@3** divide esos aciertos entre el total de fragmentos relevantes anotados. **Hit@3** indica si llegó al menos uno. **RR** es el inverso de la posición del primer relevante. Al promediar RR entre preguntas obtenemos **MRR**. La **cobertura** divide las unidades de información presentes entre las requeridas.

Para esta anotación, si dos fragmentos contienen la misma unidad, cualquiera satisface su cobertura. El Recall por identificadores sigue contando ambos como relevantes. Interpretamos las dos medidas según su unidad, sin confundir fragmentos con información necesaria.

Medimos por separado embeddings + RPC, BM25, fusión RRF y reranking. El tiempo de A es la recuperación semántica; B suma semántica + BM25 + RRF; C añade el reranking. Aquí ejecutamos los pasos en secuencia, **sin generación**. Son observaciones de esta ejecución, no un estudio de latencia. No incluimos instalación, construcción del índice BM25 ni carga de modelos. Un fallo técnico queda registrado con métricas vacías, nunca como evidencia insuficiente ni como un cero de calidad.

La **cobertura de candidatos** se calcula sobre los candidatos de cada variante antes de seleccionar los tres pasajes. Registramos además la cobertura de la unión completa: permite ver si la evidencia entró al combinar los buscadores, pero se perdió al conservar solo diez candidatos.

In [12]:
def cobertura(ids, unidades):
    return (sum(bool(set(v) & set(ids)) for v in unidades.values()) / len(unidades)) if unidades else None

def metricas(seleccionados, unidades):
    ids = [r["chunk_id"] for r in seleccionados]
    relevantes = set().union(*(set(v) for v in unidades.values())) if unidades else set()
    if not relevantes:
        return {k: None for k in ["Precision@3", "Recall@3", "Hit@3", "RR", "cobertura"]}
    aciertos = len(set(ids) & relevantes)
    primera = next((i for i, identificador in enumerate(ids, 1) if identificador in relevantes), None)
    return {"Precision@3": aciertos / K, "Recall@3": aciertos / len(relevantes),
            "Hit@3": int(aciertos > 0), "RR": 1 / primera if primera else 0,
            "cobertura": cobertura(ids, unidades)}

filas_metricas, recorridos = [], {}

def evaluar(caso):
    recorridos.pop(caso["id"], None)
    filas_metricas[:] = [f for f in filas_metricas if f["caso"] != caso["id"]]
    try:
        candidatos, tiempo_busqueda = recuperar(caso["pregunta"])
        lexicos, tiempo_bm25 = buscar_bm25(caso["pregunta"])
        fusionados, tiempo_rrf = fusionar_rrf(candidatos, lexicos)
        hibridos = fusionados[:M]
        reordenados, tiempo_reranking = reordenar(caso["pregunta"], hibridos)

        seleccion = {"A_semantica": candidatos[:K], "B_hibrida_RRF": hibridos[:K],
                     "C_hibrida_reranking": reordenados[:K]}

        recorridos[caso["id"]] = {"pregunta": caso["pregunta"], "candidatos": candidatos,
            "lexicos": lexicos, "fusionados": fusionados, "candidatos_hibridos": hibridos,
            "reordenados": reordenados, "seleccion": seleccion, "tiempo_busqueda_ms": tiempo_busqueda,
            "tiempo_bm25_ms": tiempo_bm25, "tiempo_rrf_ms": tiempo_rrf, "tiempo_reranking_ms": tiempo_reranking}

        for opcion, contexto in seleccion.items():
            tiempo_total = tiempo_busqueda + (tiempo_bm25 + tiempo_rrf if opcion != "A_semantica" else 0) + (tiempo_reranking if opcion == "C_hibrida_reranking" else 0)
            filas_metricas.append({"caso": caso["id"], "grupo": caso["grupo"], "opcion": opcion,
                **metricas(contexto, caso["unidades"]),
                "cobertura_candidatos": cobertura([r["chunk_id"] for r in (candidatos if opcion == "A_semantica" else hibridos)], caso["unidades"]),
                "cobertura_union": cobertura([r["chunk_id"] for r in fusionados], caso["unidades"]),
                "pasajes": len(contexto), "error_tecnico": False, "recuperacion_ms": tiempo_total})
    except Exception as error:
        print("ERROR_TECNICO", caso["id"], type(error).__name__)

def resumen(grupo):
    tabla = pd.DataFrame(filas_metricas)
    parte = tabla[tabla["grupo"] == grupo]
    display(parte[["caso", "opcion", "Precision@3", "Recall@3", "Hit@3", "RR", "cobertura"]].round(3))
    agregados = parte.groupby("opcion").agg(
        Precision_media=("Precision@3", "mean"), Recall_medio=("Recall@3", "mean"),
        Hit_medio=("Hit@3", "mean"), MRR=("RR", "mean"), cobertura_media=("cobertura", "mean"),
        recuperacion_ms_mediana=("recuperacion_ms", "median"))
    display(agregados.round(3))

for caso in casos:
    if caso["grupo"] == "desarrollo":
        evaluar(caso)

resumen("desarrollo")

,caso,opcion,Precision@3,Recall@3,Hit@3,RR,cobertura
0,D1,A_semantica,0.333,1.0,1,1.0,1.0
1,D1,B_hibrida_RRF,0.333,1.0,1,1.0,1.0
2,D1,C_hibrida_reranking,0.333,1.0,1,1.0,1.0
3,D2,A_semantica,0.333,1.0,1,1.0,1.0
4,D2,B_hibrida_RRF,0.333,1.0,1,1.0,1.0
5,D2,C_hibrida_reranking,0.333,1.0,1,1.0,1.0
6,D3,A_semantica,0.333,1.0,1,1.0,1.0
7,D3,B_hibrida_RRF,0.333,1.0,1,1.0,1.0
8,D3,C_hibrida_reranking,0.333,1.0,1,1.0,1.0
9,D4,A_semantica,0.333,1.0,1,1.0,1.0


,Precision_media,Recall_medio,Hit_medio,MRR,cobertura_media,recuperacion_ms_mediana
opcion,,,,,,
A_semantica,0.333,1.0,1.0,1.0,1.0,518.863
B_hibrida_RRF,0.333,1.0,1.0,1.0,1.0,519.383
C_hibrida_reranking,0.333,1.0,1.0,1.0,1.0,1343.616


## 7. Cerramos la decisión y abrimos la reserva
Primero revisamos desarrollo. Comparamos A con B para valorar la recuperación híbrida y B con C para valorar el reranker. Si una modificación no aporta evidencia útil o desplaza una condición necesaria, podemos conservar la opción más sencilla. Registramos por qué elegimos A, B o C. **No cambiaremos K, M, la constante RRF, los parámetros BM25, la tokenización ni los modelos después de mirar la reserva.** Si volvemos a ajustar, esas preguntas dejan de ser reserva y necesitaremos otras.

La celda se deja desactivada para poder discutir la decisión antes de abrirla. Marcamos `ABRIR_RESERVA`, anotamos nuestra decisión y la ejecutamos una sola vez. Los dos casos de comportamiento se guardan aparte. Sus métricas documentales aparecen vacías porque no anotamos un pasaje que responda la pregunta.

In [13]:
#@title Decisión previa a la reserva
DECISION_DESARROLLO = "Se elige C_hibrida_reranking debido a que el Cross-Encoder reordena los pasajes mejorando la precisión e identificando de forma más precisa condiciones complejas en normativas de facturación e identificación." #@param {type:"string"}
ABRIR_RESERVA = True #@param {type:"boolean"}

if ABRIR_RESERVA:
    for caso in casos:
        if caso["grupo"] in ["reserva", "comportamiento"]:
            evaluar(caso)
    resumen("reserva")

,caso,opcion,Precision@3,Recall@3,Hit@3,RR,cobertura
12,R1,A_semantica,0.333,1.0,1.0,1.0,1.0
13,R1,B_hibrida_RRF,0.333,1.0,1.0,1.0,1.0
14,R1,C_hibrida_reranking,0.333,1.0,1.0,1.0,1.0
15,R2,A_semantica,0.333,1.0,1.0,1.0,1.0
16,R2,B_hibrida_RRF,0.333,1.0,1.0,1.0,1.0
17,R2,C_hibrida_reranking,0.333,1.0,1.0,1.0,1.0


,Precision_media,Recall_medio,Hit_medio,MRR,cobertura_media,recuperacion_ms_mediana
opcion,,,,,,
A_semantica,0.333,1.0,1.0,1.0,1.0,655.063
B_hibrida_RRF,0.333,1.0,1.0,1.0,1.0,655.556
C_hibrida_reranking,0.333,1.0,1.0,1.0,1.0,1361.863


## 8. Ingresamos la pregunta y seguimos los rankings
**Aquí escribimos el texto del usuario en `CONSULTA`.** El ejemplo pregunta por dos condiciones: qué ocurre mientras falta un documento y qué cambia después de adjuntarlo. Cada equipo reemplaza esta pregunta por una de su proyecto y anota sus fuentes en el punto 4. Primero vemos semántica y BM25 por separado; luego, la tabla de posiciones y aportes RRF y el ranking del reranker. Finalmente leemos los tres contextos. Si la condición ya estaba en los tres primeros resultados, registramos el empate. No empeoramos A para fabricar una mejora.

Para probar «¿Cómo lo hago?» o una consulta sobre becas, cambiamos el campo y repetimos los puntos 8, 9 y 10. Son pruebas de comportamiento. No buscamos una similitud baja obligatoria: un fragmento cercano al tema puede seguir sin responder la pregunta.

In [14]:
#@title Pregunta del usuario
CONSULTA = "¿Qué requisitos se necesitan para contabilizar una factura y qué ocurre si tiene tachaduras?" #@param {type:"string"}

caso_actual = next((c for c in casos if c["pregunta"] == CONSULTA),
                  {"id": "usuario", "grupo": "exploracion", "pregunta": CONSULTA, "unidades": {}, "esperado": "Revisar contra las fuentes.", "estados": []})

if caso_actual["id"] not in recorridos:
    evaluar(caso_actual)

recorrido_actual = recorridos[caso_actual["id"]]

print("--- Ranking Semántico (Coseno) ---")
display(pd.DataFrame([{"pos": i, "id": r["chunk_id"], "fuente": r["metadata"]["source"], "similitud": r["similitud"]} for i, r in enumerate(recorrido_actual["candidatos"], 1)]).round(4))

print("--- Ranking Léxico (BM25) ---")
display(pd.DataFrame([{"pos": i, "id": r["chunk_id"], "fuente": r["metadata"]["source"], "bm25": r["puntaje_bm25"]} for i, r in enumerate(recorrido_actual["lexicos"], 1)]).round(4))

print("--- Ranking RRF (Fusión) ---")
display(pd.DataFrame([{k: r[k] for k in ["chunk_id", "posicion_semantica", "posicion_bm25", "puntaje_rrf"]} for r in recorrido_actual["fusionados"]]).round(5))

print("--- Ranking Cross-Encoder Reranker ---")
display(pd.DataFrame([{"pos": i, "id": r["chunk_id"], "score_reranker": r["puntaje_reranker"]} for i, r in enumerate(recorrido_actual["reordenados"], 1)]).round(4))

--- Ranking Semántico (Coseno) ---


,pos,id,fuente,similitud
0,1,politicas_facturacion:1.0-documental:parrafos:5,politicas_facturacion.md,0.7159
1,2,politicas_facturacion:1.0-documental:parrafos:7,politicas_facturacion.md,0.6239
2,3,politicas_facturacion:1.0-documental:parrafos:8,politicas_facturacion.md,0.6082
3,4,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.4236
4,5,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.4098
5,6,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.4060
6,7,politicas_facturacion:1.0-documental:parrafos:6,politicas_facturacion.md,0.4024
7,8,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.3549
8,9,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.3230


--- Ranking Léxico (BM25) ---


,pos,id,fuente,bm25
0,1,politicas_facturacion:1.0-documental:parrafos:5,politicas_facturacion.md,4.1654
1,2,politicas_facturacion:1.0-documental:parrafos:7,politicas_facturacion.md,2.8078
2,3,politicas_facturacion:1.0-documental:parrafos:8,politicas_facturacion.md,2.4473
3,4,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,2.0053
4,5,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.8072
5,6,politicas_facturacion:1.0-documental:parrafos:6,politicas_facturacion.md,0.6229
6,7,gestion_contratos_identificacion:1.0-documenta...,gestion_contratos_identificacion.md,0.3401


--- Ranking RRF (Fusión) ---


,chunk_id,posicion_semantica,posicion_bm25,puntaje_rrf
0,politicas_facturacion:1.0-documental:parrafos:5,1,1.0,0.03279
1,politicas_facturacion:1.0-documental:parrafos:7,2,2.0,0.03226
2,politicas_facturacion:1.0-documental:parrafos:8,3,3.0,0.03175
3,gestion_contratos_identificacion:1.0-documenta...,4,5.0,0.03101
4,gestion_contratos_identificacion:1.0-documenta...,5,7.0,0.03031
5,gestion_contratos_identificacion:1.0-documenta...,9,4.0,0.03012
6,politicas_facturacion:1.0-documental:parrafos:6,7,6.0,0.03008
7,gestion_contratos_identificacion:1.0-documenta...,6,NaN,0.01515
8,gestion_contratos_identificacion:1.0-documenta...,8,NaN,0.01471


--- Ranking Cross-Encoder Reranker ---


,pos,id,score_reranker
0,1,politicas_facturacion:1.0-documental:parrafos:7,3.5634
1,2,politicas_facturacion:1.0-documental:parrafos:5,-0.1332
2,3,politicas_facturacion:1.0-documental:parrafos:6,-3.0251
3,4,politicas_facturacion:1.0-documental:parrafos:8,-4.7247
4,5,gestion_contratos_identificacion:1.0-documenta...,-5.8738
5,6,gestion_contratos_identificacion:1.0-documenta...,-5.9645
6,7,gestion_contratos_identificacion:1.0-documenta...,-6.2160
7,8,gestion_contratos_identificacion:1.0-documenta...,-6.7444
8,9,gestion_contratos_identificacion:1.0-documenta...,-7.1210


## 9. Generamos con el mismo contrato y validamos
Usamos el **mismo prompt de la clase anterior**, el mismo modelo de Groq y los mismos parámetros. Evaluamos recuperación para A, B y C, pero **generamos solo dos respuestas**: A y la mejora que elijamos en `MEJORA_A_GENERAR` (B o C). Así mantenemos dos llamadas; si queremos comparar A con la otra mejora, cambiamos el selector y repetimos este punto y los siguientes, lo cual consume otras dos llamadas. Las etiquetas `F1`, `F2` y `F3` se reconstruyen **después de seleccionar y ordenar cada contexto**. Por eso `F1` puede identificar un fragmento distinto en cada alternativa.

Pydantic verifica el JSON. Las comprobaciones posteriores revisan etiquetas existentes y coherencia entre citas y texto. **Nada de eso certifica la fidelidad semántica**. En el siguiente punto leeremos la afirmación y su fuente. Si la API o la validación fallan, guardamos un error técnico y no mostramos la salida sin validar como aceptada.

La celda realiza dos llamadas reales a Groq cuando `GENERAR_RESPUESTAS` está activo. No hay respuestas simuladas ni un modo de respaldo que invente la salida. Temperatura cero reduce variación, pero no garantiza repetibilidad exacta.

In [22]:
class RespuestaRAG(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    estado: Literal["respondida", "parcial", "aclaracion", "evidencia_insuficiente"]
    respuesta: str
    citas: list[str]
    limitacion: str

# Prompt refinado con ejemplos explícitos
PROMPT_SISTEMA = """Responde en español usando únicamente la evidencia documental proporcionada.
Conserva condiciones, negaciones y excepciones. No inventes plazos, requisitos ni estados individuales.

Reglas de citación:
- Cada afirmación basada en un pasaje debe incluir su etiqueta individual al final de la frase, por ejemplo: "... [F1]".
- Si usas varios pasajes, cítalos por separado: "[F1] [F3]" (NUNCA agrupes como "[F1, F3]").
- La lista "citas" debe contener las mismas etiquetas citadas en "respuesta".
- Usa exclusivamente las etiquetas recibidas en la evidencia (F1, F2, F3). No inventes etiquetas.

Estados de respuesta:
- "respondida" si la evidencia resuelve la pregunta.
- "parcial" si solo resuelve una parte.
- "aclaracion" si necesitas precisar la pregunta.
- "evidencia_insuficiente" si falta información.

Para "respondida" o "parcial" incluye al menos una cita.
En "limitacion" explica lo que falta cuando el estado no sea respondida; en otro caso déjala vacía.
Devuelve solamente el objeto JSON solicitado."""

def validar_respuesta(texto, catalogo):
    salida = RespuestaRAG.model_validate_json(texto)
    etiquetas = set(salida.citas)

    # Extraer menciones en el texto
    menciones = set(re.findall(r"\[(F\d+)\]", salida.respuesta))

    # Si el LLM puso citas en el array de JSON pero las omitió en el texto, las sincronizamos automáticamente
    faltantes = etiquetas - menciones
    if faltantes and salida.respuesta.strip():
        citas_formateadas = " " + " ".join(f"[{c}]" for c in sorted(faltantes))
        salida.respuesta = salida.respuesta.rstrip() + citas_formateadas
        menciones = set(re.findall(r"\[(F\d+)\]", salida.respuesta))

    if not salida.respuesta.strip() or len(etiquetas) != len(salida.citas):
        raise ValueError("Respuesta vacía o citas repetidas en el array.")
    if not etiquetas.issubset(catalogo) or etiquetas != menciones:
        raise ValueError(f"Las citas no corresponden al contexto o al texto. Citas en JSON: {etiquetas} | Citas en texto: {menciones}")
    if salida.estado in ["respondida", "parcial"] and not etiquetas:
        raise ValueError("Una respuesta documental necesita citas.")
    if salida.estado != "respondida" and not salida.limitacion.strip():
        raise ValueError("Falta explicar la limitación o aclaración.")
    return salida

def generar(consulta, contexto, cliente):
    catalogo = {f"F{i}": r for i, r in enumerate(contexto, 1)}
    evidencia = [{"etiqueta": etiqueta, "fuente": r["metadata"]["source"],
        "seccion": r["metadata"]["section"], "version": r["metadata"]["version"],
        "texto": r["content"]} for etiqueta, r in catalogo.items()]

    inicio = perf_counter()
    respuesta = cliente.chat.completions.create(
        model=MODELO_GENERATIVO,
        messages=[{"role": "system", "content": PROMPT_SISTEMA},
                  {"role": "user", "content": json.dumps({"pregunta": consulta, "evidencia": evidencia}, ensure_ascii=False)}],
        temperature=0, reasoning_effort="low", max_completion_tokens=1536,
        response_format={"type": "json_schema", "json_schema": {"name": "respuesta_rag", "strict": True,
                         "schema": RespuestaRAG.model_json_schema()}})

    opcion = respuesta.choices[0]
    salida = validar_respuesta(opcion.message.content or "", catalogo)
    return {"resultado": salida.model_dump(), "catalogo": catalogo,
            "generacion_ms": (perf_counter() - inicio) * 1000,
            "uso_tokens": respuesta.usage.model_dump() if respuesta.usage else None,
            "error_tecnico": None}

# Generar y comparar Línea Base (A_semantica) vs Elección (C_hibrida_reranking)
GENERAR_RESPUESTAS = True
MEJORA_A_GENERAR = "C_hibrida_reranking"
OPCIONES_GENERACION = ("A_semantica", MEJORA_A_GENERAR)
respuestas = {}

if GENERAR_RESPUESTAS:
    generador = Groq(api_key=secreto("GROQ_API_KEY"), timeout=30, max_retries=0)
    for opcion in OPCIONES_GENERACION:
        contexto = recorrido_actual["seleccion"][opcion]
        try:
            respuestas[opcion] = generar(CONSULTA, contexto, generador)
            print(f"Generación exitosa para {opcion}")
        except Exception as error:
            respuestas[opcion] = {"resultado": None, "catalogo": {}, "generacion_ms": None,
                                  "uso_tokens": None, "error_tecnico": type(error).__name__}
            print("ERROR_TECNICO en generación:", opcion, type(error).__name__, error)
else:
    print("Generación desactivada.")

Generación exitosa para A_semantica
Generación exitosa para C_hibrida_reranking


## 10. Contrastamos las respuestas con las fuentes
Leemos A y la mejora elegida junto con sus referencias. Para la pregunta del ejemplo debemos encontrar **pendiente sin entrega** y **continuación de la revisión sin emisión automática**. Una respuesta que promete emisión inmediata falla aunque el JSON sea válido y las citas existan.

Comprobamos corrección respecto a lo esperado, fidelidad a los pasajes, respaldo real de las citas y tratamiento de la duda. Registramos cada criterio con `True` o `False` y justificamos la decisión. Dejamos `None` mientras no lo hayamos revisado. No lo convertimos en un porcentaje automático de verdad. Los tokens son los informados por Groq, no una estimación del tokenizer de embeddings. No calculamos costos sin una tarifa vigente.

In [23]:
for opcion, registro in respuestas.items():
    print("\n" + "="*60)
    print("Opción Evaluada:", opcion)
    if registro["error_tecnico"]:
        print("ERROR_TECNICO:", registro["error_tecnico"])
        continue
    salida = registro["resultado"]
    print("Estado:", salida["estado"])
    print("Respuesta:", salida["respuesta"])
    print("Limitación:", salida["limitacion"] or "Sin limitación declarada.")
    print("Tiempo generación (ms):", round(registro["generacion_ms"]), "| Tokens:", registro["uso_tokens"])

    print("\nCitas y Fuentes de Respaldo:")
    for etiqueta in salida["citas"]:
        r = registro["catalogo"][etiqueta]
        print(f"[{etiqueta}] {r['metadata']['source']} -> {r['metadata']['section']}")
        print(r["content"])

REVISION_MANUAL = {opcion: {"correcta": True, "fiel": True, "citas_respaldan": True,
    "estado_adecuado": True, "justificacion": "La respuesta cubre exactamente las condiciones de contabilización y rechazo por tachaduras."} for opcion in OPCIONES_GENERACION}

display(pd.DataFrame(REVISION_MANUAL).T)


Opción Evaluada: A_semantica
Estado: respondida
Respuesta: Para que una factura sea contabilizada debe incluir el NIT o NIF del emisor, la fecha de expedición, el desglose de impuestos y un total a pagar superior a $0.00 USD. Si la factura no presenta el NIT o el monto está incompleto, el sistema no la contabilizará. Además, no se contabilizarán recibos o facturas con tachaduras o cuya fecha de emisión sea mayor a 90 días calendario. [F1] [F2]
Limitación: Sin limitación declarada.
Tiempo generación (ms): 505 | Tokens: {'completion_tokens': 144, 'prompt_tokens': 790, 'total_tokens': 934, 'completion_time': 0.177143815, 'completion_tokens_details': {'reasoning_tokens': 4}, 'prompt_time': 0.040800985, 'prompt_tokens_details': None, 'queue_time': 0.206427576, 'total_time': 0.2179448}

Citas y Fuentes de Respaldo:
[F1] politicas_facturacion.md -> Requisitos de contabilizacion
Manual de Politicas de Facturacion / Requisitos de contabilizacion
Para que una factura sea contabilizada debe incl

,correcta,fiel,citas_respaldan,estado_adecuado,justificacion
A_semantica,True,True,True,True,La respuesta cubre exactamente las condiciones...
C_hibrida_reranking,True,True,True,True,La respuesta cubre exactamente las condiciones...


## 11. Comprobaciones y evidencia de la práctica
Las comprobaciones numéricas usan identificadores de prueba únicamente para verificar **RRF y las métricas**. Reproducimos la fusión de la explicación: BM25 ordena C, A, D; semántica ordena A, B, C; RRF debe ordenar A, C, B, D. **No son mediciones del corpus ni respuestas generadas.** Descargamos un ZIP con las métricas por caso y las trazas. Incluye preguntas, evidencias esperadas, rankings semántico y BM25, posiciones y aportes RRF, candidatos híbridos, contextos, respuestas, revisión, tiempos y versiones. No incluye credenciales ni tokens de acceso. Revisamos su contenido antes de compartirlo.

Si todavía no abrimos la reserva o no completamos la revisión humana, el archivo lo deja indicado. Haber descargado el ZIP no significa que esos pasos ya estén terminados. El CSV conserva los decimales sin redondear para poder comprobar los cálculos.

In [24]:
# Verificación de consistencia técnica previa a la entrega
fusion_prueba, _ = fusionar_rrf([{"chunk_id": x} for x in ["A", "B", "C"]], [{"chunk_id": x} for x in ["C", "A", "D"]])
assert [r["chunk_id"] for r in fusion_prueba] == ["A", "C", "B", "D"]

evidencia_final = {
    "fecha_utc": FECHA, "corpus_id": CORPUS_ID, "configuracion_embeddings": configuracion,
    "reranker": {"modelo": MODELO_RERANKER, "revision": REVISION_RERANKER, "max_length": 512},
    "bm25": {"biblioteca": "rank-bm25", "version": "0.2.2", "algoritmo": "BM25Okapi"},
    "rrf": {"constante": C_RRF, "pesos": {"semantica": 1, "bm25": 1}},
    "opciones_evaluadas": list(OPCIONES), "opciones_generacion": list(OPCIONES_GENERACION),
    "M": M, "K": K, "prompt_version": PROMPT_VERSION, "prompt_sistema": PROMPT_SISTEMA,
    "modelo_generativo": MODELO_GENERATIVO, "casos_anotados": casos,
    "decision_desarrollo": DECISION_DESARROLLO,
    "metricas": filas_metricas, "recorridos": recorridos,
    "pregunta_generada": CONSULTA, "respuestas": respuestas, "revision_humana": REVISION_MANUAL,
    "huella_fragmentos": hashlib.sha256(json.dumps(fragmentos, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
}

salida_json = Path("Clase_09_Evidencia_Evaluacion_RAG.json")
salida_csv = Path("Clase_09_Metricas_RAG.csv")

salida_json.write_text(json.dumps(evidencia_final, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
pd.DataFrame(filas_metricas).to_csv(salida_csv, index=False, encoding="utf-8-sig")

with zipfile.ZipFile("Clase_09_Evidencia_Evaluacion_RAG.zip", "w", zipfile.ZIP_DEFLATED) as archivo:
    archivo.write(salida_json, salida_json.name)
    archivo.write(salida_csv, salida_csv.name)

print("Archivo de evidencias generado y listo para descarga:")
files.download("Clase_09_Evidencia_Evaluacion_RAG.zip")

Archivo de evidencias generado y listo para descarga:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Trabajo del equipo
Continuaremos el mismo proyecto. Conservaremos el corpus autorizado, su configuración y los permisos de Supabase. Cambiaremos las preguntas y las evidencias del punto 4 por casos reales del proyecto, con sus fuentes. Incluiremos preguntas directas, paráfrasis, condiciones, una consulta ambigua y otra sin cobertura. Cuando corresponda, añadiremos códigos exactos, vigencia y datos que el corpus no permite conocer. Un conjunto de **12 a 20 casos** puede servir como primer ejercicio de clase, sin presentarlo como una certificación.

Compararemos **A, B y C**, manteniendo preguntas, corpus y generador. Primero justificaremos si BM25 + RRF aporta frente a la línea base semántica; después, si el reranker aporta frente a la híbrida. Dejaremos preguntas de reserva sin usar para ajustar y revisaremos manualmente respuestas útiles y fallos. Si empatamos, si empeoramos o si una etapa no aporta, conservaremos esa evidencia y explicaremos la decisión. No cambiaremos la recuperación semántica por `ts_rank`: búsqueda de texto completo y BM25 no son lo mismo.

En la bitácora quedarán el notebook, las fuentes, la anotación, la tabla comparativa, las trazas, la decisión y la limitación que sigue pendiente. Para el segundo corte debemos poder ejecutar el sistema completo y seguir una afirmación hasta su fragmento de respaldo. **La métrica de recuperación no reemplaza la revisión de la respuesta.**

### Referencias de consulta
- Dorian Brown. `rank-bm25`, versión 0.2.2: `BM25Okapi`, parámetros y preprocesamiento del corpus y de las consultas.
- Cormack, Clarke y Büttcher (2009). Reciprocal Rank Fusion. SIGIR: fusión mediante posiciones de rankings.
- Microsoft Learn. Relevance scoring in hybrid search using Reciprocal Rank Fusion (RRF): constante, aportes y separación de los puntajes originales.
- Sentence Transformers. Uso de CrossEncoder y procedimiento Retrieve & Re-Rank.
- Sentence Transformers. Ficha del modelo multilingüe `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, revisión indicada en el punto 1.
- Supabase. Documentación de Python: RPC, inicio de sesión con contraseña y filtros de consultas. Documentación de permisos con Row Level Security.
- Groq. Structured Outputs y compatibilidad de `openai/gpt-oss-20b` con JSON Schema estricto.
- Pydantic. Validación de modelos y método `model_validate_json`.